# Riemann + manifold gate

Эмбеддинг сессии — смесь обучаемых эмбеддингов ближайших train-людей на многообразии:
$\alpha_k = \mathrm{softmax}_k(-\|f - c_k\|^2/\tau)$, $e = \sum_k \alpha_k e_k$. Через FiLM он подстраивает классификатор.
Код: `bci/models/gated.py`.

In [ ]:
import sys; sys.path.append("..")
from bci.data import load_split, concat
from bci.evaluation import evaluate, cross_validation_summary, compare_with_baselines

train, val, test = (load_split(s, "../data") for s in ["train", "val", "test"])
dev = concat(train, val)  # для кросс-валидации; test только в самом конце
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from bci.models import gated

PARAMS = dict(epochs=150, lr=1e-3, hide_own=0.5, dim=32, emb=8, band=(4, 40), prior_weight=5)
fit = lambda tr: gated.fit(tr, **PARAMS)

## Люди на многообразии
Точка — сессия (её опорная точка в конце записи), цвет — участник. Гейт работает, если сессии одного человека лежат рядом.

In [ ]:
from sklearn.decomposition import PCA
from bci.riemann import RiemannFeatures

_, fp = RiemannFeatures().fit_transform(dev)
last = [idx[-1] for idx in dev.streams()]
z = PCA(2).fit_transform(fp[last])
plt.scatter(*z.T, c=pd.factorize(dev.subject[last])[0], cmap="tab10", s=60)
for (a, b), s in zip(z, dev.subject[last]):
    plt.annotate(s.split("/")[-1], (a, b), fontsize=8)
plt.title("сессии на многообразии");

## Сколько эпох
Кривая val (новые люди) не должна заметно падать к пунктиру. Если падает, уменьшите `epochs`.

In [ ]:
model = gated.fit(train, **{**PARAMS, "epochs": 300}, val=val)
ax = pd.Series(model.history).rolling(25, min_periods=1).mean().plot(title="val session_mean (сглажено)")
ax.axvline(PARAMS["epochs"], ls="--", c="gray")
print(f"τ = {model.net.log_tau.exp().item():.1f}")

## Кросс-валидация на train + val
По ней сравниваем гипотезы: новые люди (каждый по очереди в тесте) и поздние сессии знакомых людей.

In [ ]:
cross_validation_summary(fit, dev)

Узнаёт ли гейт знакомого человека по поздней сессии? Доля окон, где самый большой вес у него самого.

In [ ]:
from bci.evaluation import later_sessions

_, early, late = next(later_sessions(dev))
model = fit(early)
_, fp_late = model.features.transform(late)
with torch.no_grad():
    picked = model.net.gate(torch.as_tensor(fp_late, dtype=torch.float32)).argmax(-1).numpy()
hit = np.array(model.subjects)[picked] == late.subject
print(f"гейт выбрал самого человека в {hit.mean():.0%} окон (случайно: {1 / len(model.subjects):.0%})")

## Официальный test
Обучение на train, одна оценка на test, рядом опубликованные бейзлайны.

In [ ]:
compare_with_baselines({"Riemann + manifold gate": evaluate(fit(train), test)})